### Test with AutoGluon 

In [1]:
import pandas as pd
import numpy as np
print("numpy version:", np.__version__) # 1.6.2 needed for autogluon
from autogluon.tabular import TabularPredictor
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

numpy version: 1.26.4


c:\Users\Marte\Documents\SCHOOL\2024_2026_KUL\2025-2026\SEMESTER 2\Advanced Analytics in a Big Data World\Assignment1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
transactions = pd.read_csv(r"original_data\transactions_2016_2017.csv")
train_labels = pd.read_csv(r"original_data\customer_clv_train.csv")
test_labels =  pd.read_csv(r"original_data\customer_clv_test.csv")     

C:\Users\Marte\AppData\Local\Temp\ipykernel_14064\2056241763.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  transactions = pd.read_csv(r"C:\Users\Marte\Documents\SCHOOL\2024_2026_KUL\2025-2026\SEMESTER 2\Advanced Analytics in a Big Data World\Assignment1\BigData_assignment1\data\original_data\transactions_2016_2017.csv")


In [3]:
transactions.shape
transactions["cust_id"].nunique() # 145 739 unieke klanten

145739

In [ ]:
features = pd.read_csv(r"data\features_enriched.csv")
features.shape # 145 739 rijen (komt overeen met aantal unieke klanten in transactions)
features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145739 entries, 0 to 145738
Data columns (total 87 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   cust_id                                145739 non-null  object 
 1   n_items                                145739 non-null  int64  
 2   n_sales                                145739 non-null  int64  
 3   n_returns                              145739 non-null  int64  
 4   first_purchase                         145739 non-null  object 
 5   last_purchase                          145739 non-null  object 
 6   n_items_non_returned                   145739 non-null  float64
 7   n_sales_non_returned                   145739 non-null  float64
 8   total_revenue_net                      145739 non-null  float64
 9   total_discount                         145739 non-null  float64
 10  total_revenue_gross                    145739 non-null  

In [6]:
print(train_labels.head())
print(test_labels.head())

            cust_id  revenue_2018_2019
0  klantwj2374mzmab             209.85
1  a63atwr2ig2jfprr              82.93
2  zr7ihbfbi6gcy2tz              89.95
3  dt7cthjqnjmkbiu6               0.00
4  etcrrgcbrzfovyzj               0.00
            cust_id
0  2dfoualegmpt6x2h
1  d2q2stjpnzld7a4r
2  cojscuqlpylhclv2
3  vntezlhi2ryvxk6m
4  jgy4ytjkdr2b75wf


In [7]:
# Mergen met revenue 2018-2019 (om de target variable te hebben)
master_train = train_labels.merge(features, on="cust_id", how="left").fillna(0) #train labels als basis 
master_train.head()

,cust_id,revenue_2018_2019,n_items,n_sales,n_returns,first_purchase,last_purchase,n_items_non_returned,n_sales_non_returned,total_revenue_net,...,prod_print_n_unique,prod_comfort_sole_dominant_share,prod_comfort_sole_n_unique,prod_comfort_wear_dominant_share,prod_comfort_wear_n_unique,prod_clasp_dominant_share,prod_clasp_n_unique,prob_alive,pred_num_purchases,expected_avg_sales
0,klantwj2374mzmab,209.85,2,1,1,2016-01-01,2016-01-01,1.0,1.0,83.30,...,0.0,0.0,0.0,0.0,0.0,0.500000,2.0,1.000000,0.011254,0.000000
1,a63atwr2ig2jfprr,82.93,4,2,3,2016-01-01,2017-06-21,1.0,1.0,53.96,...,1.0,0.0,0.0,1.0,1.0,0.666667,2.0,0.724322,0.026478,63.250871
2,zr7ihbfbi6gcy2tz,89.95,2,2,0,2016-01-01,2017-01-27,2.0,2.0,83.70,...,0.0,1.0,1.0,1.0,1.0,0.500000,2.0,0.658666,0.024078,48.008387
3,dt7cthjqnjmkbiu6,0.00,4,2,1,2016-01-01,2017-05-24,3.0,2.0,350.92,...,0.0,0.0,0.0,0.0,0.0,0.500000,3.0,0.713269,0.026074,310.739524
4,etcrrgcbrzfovyzj,0.00,6,2,3,2016-01-01,2017-01-28,3.0,2.0,144.50,...,0.0,0.0,0.0,0.0,0.0,1.000000,1.0,0.659199,0.024097,72.016934


In [8]:
# Splits het VOLLEDIGE dataframe
train_set, val_set = train_test_split(
    master_train,                  # Hier geef je de hele tabel mee (77 kolommen)
    test_size=0.2, 
    random_state=42,
    stratify=(master_train["revenue_2018_2019"] > 0)
)

# Nu bevatten train_set en val_set allebei 77 kolommen
print(f"Train set vorm: {train_set.shape}")
print(f"Validation set vorm: {val_set.shape}")

Train set vorm: (93272, 88)
Validation set vorm: (23319, 88)


In [9]:
# Test set voorbereiden (features mergen met test_labels)
test_set_final = test_labels.merge(features, on="cust_id", how="left").fillna(0)

# Bewaar de cust_id's voor later (die heb je nodig voor je submission file)
test_cust_ids = test_set_final["cust_id"]

# Verwijder kolommen die het model niet mag zien (zoals de ID en de target kolom als die er per ongeluk in zit)
X_test_ag = test_set_final.drop(columns=["cust_id", "revenue_2018_2019"], errors="ignore")

In [10]:
train_ag = train_set.drop(columns=["cust_id"]).copy()
val_ag = val_set.drop(columns=["cust_id"]).copy()

In [11]:
# 3. INITIALISEER EN FIT IN ÉÉN KEER
# We gebruiken 'path' om de unieke map te forceren
predictor = TabularPredictor(
    label="revenue_2018_2019", 
    eval_metric="mean_absolute_error",
).fit(
    train_data=train_ag,
    tuning_data=val_ag,
    use_bag_holdout=True,  # We hebben al een aparte validation set, dus we hoeven geen extra holdout set te maken
    presets="best_quality",
    time_limit= 600  # 10 minuten
)

# 4. Toon het resultaat om te zien of het gelukt is
print(predictor.leaderboard(val_ag))

No path specified. Models will be saved in: "AutogluonModels\ag-20260425_082456"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.13.2
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          12
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       3.02 GB / 15.72 GB (19.2%)
Disk Space Avail:   151.49 GB / 475.28 GB (31.9%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to False. Reason: Skip dynamic_stacking when use_bag_holdout is enabled. (use_bag_holdout=True)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "c:\Users\Marte\Documents\SCHOOL\2024_2026_KUL\2025-2026\SEMESTER 2\Advanced Analytics in a Big Data World\

                     model  score_test  score_val          eval_metric  \
0      WeightedEnsemble_L3  -61.871162 -61.871162  mean_absolute_error   
1          CatBoost_BAG_L2  -61.871164 -61.871164  mean_absolute_error   
2          CatBoost_BAG_L1  -61.940182 -61.940182  mean_absolute_error   
3      WeightedEnsemble_L2  -61.940182 -61.940182  mean_absolute_error   
4        LightGBMXT_BAG_L1  -76.457967 -76.457967  mean_absolute_error   
5          LightGBM_BAG_L2  -76.503808 -76.503808  mean_absolute_error   
6          LightGBM_BAG_L1  -76.536910 -76.536910  mean_absolute_error   
7        LightGBMXT_BAG_L2  -76.578117 -76.578117  mean_absolute_error   
8   RandomForestMSE_BAG_L2  -79.716494 -79.716494  mean_absolute_error   
9   RandomForestMSE_BAG_L1  -80.991696 -80.991696  mean_absolute_error   
10    ExtraTreesMSE_BAG_L1  -81.218155 -81.218155  mean_absolute_error   

    pred_time_test  pred_time_val    fit_time  pred_time_test_marginal  \
0         3.009381       3.795198  41

In [12]:
importance = predictor.feature_importance(val_ag)
print(importance.head(20))

These features in provided data are not utilized by the predictor and will be ignored: ['monetary_per_item', 'avg_items_per_sale']
Computing feature importance via permutation shuffling for 84 features using 5000 rows with 5 shuffle sets...
	635.87s	= Expected runtime (127.17s per shuffle set)
	184.7s	= Actual runtime (Completed 5 of 5 shuffle sets)


                            importance    stddev   p_value  n  p99_high  \
recency_ratio                 0.325225  0.100762  0.000977  5  0.532694   
total_revenue_net             0.303899  0.136777  0.003831  5  0.585524   
prob_alive                    0.276692  0.135227  0.005110  5  0.555126   
pred_num_purchases            0.267828  0.106309  0.002443  5  0.486721   
n_sales                       0.233568  0.082516  0.001595  5  0.403470   
avg_days_between_orders       0.231933  0.022124  0.000010  5  0.277486   
purchase_frequency            0.195632  0.055387  0.000695  5  0.309674   
avg_days_between_purchases    0.194705  0.069132  0.001624  5  0.337049   
customer_lifetime_days        0.191969  0.070226  0.001813  5  0.336566   
n_items                       0.146699  0.033819  0.000316  5  0.216332   
last_purchase                 0.133278  0.078662  0.009646  5  0.295245   
max_days_between_orders       0.120016  0.023966  0.000181  5  0.169363   
avg_revenue_per_day_total

In [ ]:
# Voorspel op de validatieset
y_val_true = val_set["revenue_2018_2019"]
y_val_pred = predictor.predict(val_ag.drop(columns=["revenue_2018_2019"]))

# Bereken metrieken
mae_val = mean_absolute_error(y_val_true, y_val_pred)
spearman_val, _ = spearmanr(y_val_true, y_val_pred)

autogluon_metrics = pd.DataFrame([
    {
        "model_name": "AutoGluon",
        "AUC": np.nan,
        "MAE": mae_val,
        "Spearman": spearman_val
    }
])
autogluon_metrics.to_csv("autogluon_model_comparison.csv", index=False)

print(f"Validatie MAE: {mae_val}")
print(f"Validatie Spearman: {spearman_val}")
print("AutoGluon metrics saved to autogluon_model_comparison.csv")

Validatie MAE: 61.871162078885106
Validatie Spearman: 0.3771454133066694
AutoGluon metrics saved to autogluon_model_comparison.csv


In [14]:
# Submission maken met AutoGluon predictions op de test set

autogluon_pred_test = predictor.predict(X_test_ag)
autogluon_pred_test = np.clip(autogluon_pred_test, 0, None)

submission = pd.DataFrame({
    "cust_id": test_cust_ids,
    "predicted_revenue_2018_2019": autogluon_pred_test
})

submission_name = "submission_autogluon.csv"
submission.to_csv(submission_name, index=False)

print("Saved:", submission_name)
print("Prediction summary")
print("min :", autogluon_pred_test.min())
print("max :", autogluon_pred_test.max())
print("mean:", autogluon_pred_test.mean())

submission.head()

Saved: submission_autogluon.csv
Prediction summary
min : 0.0
max : 428.66693
mean: 25.655182


,cust_id,predicted_revenue_2018_2019
0,2dfoualegmpt6x2h,3.272733e+02
1,d2q2stjpnzld7a4r,3.859248e+01
2,cojscuqlpylhclv2,3.970849e-07
3,vntezlhi2ryvxk6m,1.208980e+02
4,jgy4ytjkdr2b75wf,1.294462e+02


In [15]:
# Log-variant voorbereiden

train_ag_log = train_ag.copy()
val_ag_log = val_ag.copy()

train_ag_log["revenue_2018_2019"] = np.log1p(train_ag_log["revenue_2018_2019"])
val_ag_log["revenue_2018_2019"] = np.log1p(val_ag_log["revenue_2018_2019"])

def safe_spearman(y_true, y_pred):
    corr, _ = spearmanr(y_true, y_pred)
    return 0.0 if pd.isna(corr) else corr

X_val_ag = val_ag.drop(columns=["revenue_2018_2019"]).copy()
y_val_true = val_ag["revenue_2018_2019"].copy()

In [16]:
# AutoGluon trainen op log1p(target)
# Verhoog time_limit als je de log-variant langer wilt laten zoeken.

predictor_log = TabularPredictor(
    label="revenue_2018_2019",
    eval_metric="mean_absolute_error",
).fit(
    train_data=train_ag_log,
    tuning_data=val_ag_log,
    use_bag_holdout=True,
    presets="best_quality",
    time_limit=3600
)

print(predictor_log.leaderboard(val_ag_log))

No path specified. Models will be saved in: "AutogluonModels\ag-20260425_083903"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.13.2
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          12
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       3.16 GB / 15.72 GB (20.1%)
Disk Space Avail:   150.90 GB / 475.28 GB (31.7%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to False. Reason: Skip dynamic_stacking when use_bag_holdout is enabled. (use_bag_holdout=True)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 3600s
AutoGluon will save models to "c:\Users\Marte\Documents\SCHOOL\2024_2026_KUL\2025-2026\SEMESTER 2\Advanced Analytics in a Big Data World

[1000]	valid_set's l1: 1.85344
[2000]	valid_set's l1: 1.84744
[3000]	valid_set's l1: 1.84482
[4000]	valid_set's l1: 1.84288
[5000]	valid_set's l1: 1.84072
[6000]	valid_set's l1: 1.84056
[7000]	valid_set's l1: 1.83844
[8000]	valid_set's l1: 1.83927
[9000]	valid_set's l1: 1.83839
[10000]	valid_set's l1: 1.83761


	-1.8562	 = Validation score   (-mean_absolute_error)
	152.12s	 = Training   runtime
	8.35s	 = Validation runtime
Fitting model: RandomForestMSE_BAG_L2 ... Training model for up to 994.99s of the 994.79s of remaining time.
	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=12, gpus=0, mem=0.4/3.5 GB
	-1.8782	 = Validation score   (-mean_absolute_error)
	441.41s	 = Training   runtime
	2.08s	 = Validation runtime
Fitting model: CatBoost_BAG_L2 ... Training model for up to 540.92s of the 540.72s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=10, gpus=0)
	-1.4401	 = Validation score   (-mean_absolute_error)
	68.41s	 = Training   runtime
	0.3s	 = Validation runtime
Fitting model: ExtraTreesMSE_BAG_L2 ... Training model for up to 471.15s of the 470.95s of remaining time.
	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=12, gpus=0, mem=0.4/3.3 GB
	-1.8831	 = Validation score  

                     model  score_test  score_val          eval_metric  \
0    NeuralNetTorch_BAG_L1   -1.432830  -1.432830  mean_absolute_error   
1      WeightedEnsemble_L2   -1.432830  -1.432830  mean_absolute_error   
2      WeightedEnsemble_L3   -1.432830  -1.432830  mean_absolute_error   
3          CatBoost_BAG_L2   -1.440132  -1.440132  mean_absolute_error   
4          CatBoost_BAG_L1   -1.449809  -1.449809  mean_absolute_error   
5   NeuralNetFastAI_BAG_L2   -1.785423  -1.785423  mean_absolute_error   
6   NeuralNetFastAI_BAG_L1   -1.793065  -1.793065  mean_absolute_error   
7           XGBoost_BAG_L1   -1.854120  -1.854120  mean_absolute_error   
8        LightGBMXT_BAG_L2   -1.855943  -1.855943  mean_absolute_error   
9          LightGBM_BAG_L2   -1.856166  -1.856166  mean_absolute_error   
10       LightGBMXT_BAG_L1   -1.856569  -1.856569  mean_absolute_error   
11         LightGBM_BAG_L1   -1.859360  -1.859360  mean_absolute_error   
12          XGBoost_BAG_L2   -1.859838

In [17]:
# Evaluatie van baseline en log-variant op originele schaal

y_val_pred_base = np.clip(np.asarray(predictor.predict(X_val_ag)), 0, None)
y_val_pred_log = np.expm1(predictor_log.predict(X_val_ag))
y_val_pred_log = np.clip(np.asarray(y_val_pred_log), 0, None)

mae_base = mean_absolute_error(y_val_true, y_val_pred_base)
sp_base, _ = spearmanr(y_val_true, y_val_pred_base)

mae_log = mean_absolute_error(y_val_true, y_val_pred_log)
sp_log, _ = spearmanr(y_val_true, y_val_pred_log)

comparison_df = pd.DataFrame([
    {"model_name": "AutoGluon_base", "MAE": mae_base, "Spearman": sp_base},
    {"model_name": "AutoGluon_log1p", "MAE": mae_log, "Spearman": sp_log}
])

comparison_df.to_csv("autogluon_log_model_comparison.csv", index=False)
comparison_df

,model_name,MAE,Spearman
0,AutoGluon_base,61.870484,0.377204
1,AutoGluon_log1p,61.856358,0.395572


In [18]:
# Zoek de beste ensemble-gewichten op de validatieset

rows = []

for weight_log in np.arange(0.0, 1.001, 0.05):
    weight_base = 1.0 - weight_log
    pred_ensemble = weight_base * y_val_pred_base + weight_log * y_val_pred_log

    rows.append({
        "weight_base": round(weight_base, 2),
        "weight_log": round(weight_log, 2),
        "MAE": mean_absolute_error(y_val_true, pred_ensemble),
        "Spearman": safe_spearman(y_val_true, pred_ensemble)
    })

ensemble_search = pd.DataFrame(rows)
best_mae_row = ensemble_search.sort_values(["MAE", "Spearman"], ascending=[True, False]).iloc[0]
best_spearman_row = ensemble_search.sort_values(["Spearman", "MAE"], ascending=[False, True]).iloc[0]

print("Best ensemble by MAE")
print(best_mae_row)
print()
print("Best ensemble by Spearman")
print(best_spearman_row)

ensemble_search.sort_values("MAE").head(10)

Best ensemble by MAE
weight_base     0.500000
weight_log      0.500000
MAE            61.788205
Spearman        0.378232
Name: 10, dtype: float64

Best ensemble by Spearman
weight_base     0.000000
weight_log      1.000000
MAE            61.856358
Spearman        0.395572
Name: 20, dtype: float64


,weight_base,weight_log,MAE,Spearman
10,0.50,0.50,61.788205,0.378232
11,0.45,0.55,61.788933,0.378317
9,0.55,0.45,61.789805,0.378084
12,0.40,0.60,61.791660,0.378350
8,0.60,0.40,61.792541,0.377956
13,0.35,0.65,61.795530,0.378348
7,0.65,0.35,61.796680,0.377765
14,0.30,0.70,61.800409,0.378371
6,0.70,0.30,61.802455,0.377573
15,0.25,0.75,61.806501,0.378435


In [19]:
# Verfijn rond de beste MAE-gewichten

start_log = max(0, best_mae_row["weight_log"] - 0.05)
end_log = min(1, best_mae_row["weight_log"] + 0.05)

fine_rows = []

for weight_log in np.arange(start_log, end_log + 0.001, 0.01):
    weight_base = 1.0 - weight_log
    pred_ensemble = weight_base * y_val_pred_base + weight_log * y_val_pred_log

    fine_rows.append({
        "weight_base": round(weight_base, 2),
        "weight_log": round(weight_log, 2),
        "MAE": mean_absolute_error(y_val_true, pred_ensemble),
        "Spearman": safe_spearman(y_val_true, pred_ensemble)
    })

ensemble_search_fine = pd.DataFrame(fine_rows)
final_ensemble_row = ensemble_search_fine.sort_values(["MAE", "Spearman"], ascending=[True, False]).iloc[0]

final_weight_base = float(final_ensemble_row["weight_base"])
final_weight_log = float(final_ensemble_row["weight_log"])
y_val_pred_ensemble = final_weight_base * y_val_pred_base + final_weight_log * y_val_pred_log

mae_ensemble = mean_absolute_error(y_val_true, y_val_pred_ensemble)
sp_ensemble, _ = spearmanr(y_val_true, y_val_pred_ensemble)

print("Final best ensemble")
print("Weight base:", final_weight_base)
print("Weight log :", final_weight_log)
print("Validation MAE     :", mae_ensemble)
print("Validation Spearman:", sp_ensemble)

Final best ensemble
Weight base: 0.49
Weight log : 0.51
Validation MAE     : 61.78816813301773
Validation Spearman: 0.3782576020685911


In [20]:
# Overzicht opslaan

comparison_with_ensemble = pd.DataFrame([
    {"model_name": "AutoGluon_base", "MAE": mae_base, "Spearman": sp_base},
    {"model_name": "AutoGluon_log1p", "MAE": mae_log, "Spearman": sp_log},
    {"model_name": "AutoGluon_base_log_ensemble", "MAE": mae_ensemble, "Spearman": sp_ensemble}
])

comparison_with_ensemble.to_csv("autogluon_log_ensemble_comparison.csv", index=False)
comparison_with_ensemble.sort_values("MAE")

,model_name,MAE,Spearman
2,AutoGluon_base_log_ensemble,61.788168,0.378258
1,AutoGluon_log1p,61.856358,0.395572
0,AutoGluon_base,61.870484,0.377204


In [ ]:
# Submission voor de log-variant

autogluon_log_pred_test = np.expm1(predictor_log.predict(X_test_ag))
autogluon_log_pred_test = np.clip(np.asarray(autogluon_log_pred_test), 0, None)

submission_log = pd.DataFrame({
    "cust_id": test_cust_ids,
    "predicted_revenue_2018_2019": autogluon_log_pred_test
})

submission_log_name = "submission_autogluon_log.csv"
submission_log.to_csv(submission_log_name, index=False)

print("Saved:", submission_log_name)
print("Prediction summary")
print("min :", autogluon_log_pred_test.min())
print("max :", autogluon_log_pred_test.max())
print("mean:", autogluon_log_pred_test.mean())

submission_log.head()

In [ ]:
# Submission voor het beste ensemble volgens validatie-MAE

autogluon_base_pred_test = np.clip(np.asarray(predictor.predict(X_test_ag)), 0, None)
autogluon_log_pred_test = np.expm1(predictor_log.predict(X_test_ag))
autogluon_log_pred_test = np.clip(np.asarray(autogluon_log_pred_test), 0, None)

autogluon_ensemble_pred_test = final_weight_base * autogluon_base_pred_test + final_weight_log * autogluon_log_pred_test
autogluon_ensemble_pred_test = np.clip(autogluon_ensemble_pred_test, 0, None)

submission_ensemble = pd.DataFrame({
    "cust_id": test_cust_ids,
    "predicted_revenue_2018_2019": autogluon_ensemble_pred_test
})

submission_ensemble_name = "submission_autogluon_log_ensemble.csv"
submission_ensemble.to_csv(submission_ensemble_name, index=False)

print("Saved:", submission_ensemble_name)
print("Prediction summary")
print("min :", autogluon_ensemble_pred_test.min())
print("max :", autogluon_ensemble_pred_test.max())
print("mean:", autogluon_ensemble_pred_test.mean())

submission_ensemble.head()

In [21]:
# Two-stage data voorbereiden: classifier voor buyer en regressor voor amount

train_cls = train_ag.copy()
val_cls = val_ag.copy()

train_cls["is_buyer"] = (train_cls["revenue_2018_2019"] > 0).astype(int)
val_cls["is_buyer"] = (val_cls["revenue_2018_2019"] > 0).astype(int)

train_cls = train_cls.drop(columns=["revenue_2018_2019"])
val_cls = val_cls.drop(columns=["revenue_2018_2019"])

train_buyers = train_ag[train_ag["revenue_2018_2019"] > 0].copy()
val_buyers = val_ag[val_ag["revenue_2018_2019"] > 0].copy()

train_buyers_log = train_buyers.copy()
val_buyers_log = val_buyers.copy()
train_buyers_log["revenue_2018_2019"] = np.log1p(train_buyers_log["revenue_2018_2019"])
val_buyers_log["revenue_2018_2019"] = np.log1p(val_buyers_log["revenue_2018_2019"])

X_val_cls = val_cls.drop(columns=["is_buyer"]).copy()
y_val_true = val_ag["revenue_2018_2019"].copy()
y_val_is_buyer = (y_val_true > 0).astype(int)

In [25]:
# AutoGluon classifier voor P(revenue > 0)

predictor_cls = TabularPredictor(
    label="is_buyer",
    problem_type="binary",
    eval_metric="roc_auc",
).fit(
    train_data=train_cls,
    tuning_data=val_cls,
    use_bag_holdout=True,
    presets="best_quality",
    time_limit=1800
)

print(predictor_cls.leaderboard(val_cls))

No path specified. Models will be saved in: "AutogluonModels\ag-20260425_102609"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.13.2
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          12
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       3.70 GB / 15.72 GB (23.5%)
Disk Space Avail:   142.47 GB / 475.28 GB (30.0%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to False. Reason: Skip dynamic_stacking when use_bag_holdout is enabled. (use_bag_holdout=True)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 1800s
AutoGluon will save models to "c:\Users\Marte\Documents\SCHOOL\2024_2026_KUL\2025-2026\SEMESTER 2\Advanced Analytics in a Big Data World

                      model  score_test  score_val eval_metric  \
0       WeightedEnsemble_L3    0.728899   0.728899     roc_auc   
1       WeightedEnsemble_L2    0.728786   0.728786     roc_auc   
2           CatBoost_BAG_L2    0.728644   0.728644     roc_auc   
3         LightGBMXT_BAG_L2    0.728521   0.728521     roc_auc   
4           LightGBM_BAG_L2    0.728431   0.728431     roc_auc   
5         LightGBMXT_BAG_L1    0.727944   0.727944     roc_auc   
6           CatBoost_BAG_L1    0.727755   0.727755     roc_auc   
7           LightGBM_BAG_L1    0.727637   0.727637     roc_auc   
8            XGBoost_BAG_L1    0.727556   0.727556     roc_auc   
9    NeuralNetFastAI_BAG_L2    0.727471   0.727471     roc_auc   
10   NeuralNetFastAI_BAG_L1    0.726580   0.726580     roc_auc   
11    ExtraTreesEntr_BAG_L2    0.723359   0.723359     roc_auc   
12    ExtraTreesGini_BAG_L2    0.721988   0.721988     roc_auc   
13    ExtraTreesGini_BAG_L1    0.720489   0.720489     roc_auc   
14    Extr

In [26]:
# AutoGluon regressor enkel op buyers, getraind op log1p(amount)

predictor_buyers = TabularPredictor(
    label="revenue_2018_2019",
    eval_metric="mean_absolute_error",
).fit(
    train_data=train_buyers_log,
    tuning_data=val_buyers_log,
    use_bag_holdout=True,
    presets="best_quality",
    time_limit=3600
)

print(predictor_buyers.leaderboard(val_buyers_log))

No path specified. Models will be saved in: "AutogluonModels\ag-20260425_105645"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.13.2
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          12
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       3.96 GB / 15.72 GB (25.2%)
Disk Space Avail:   137.96 GB / 475.28 GB (29.0%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to False. Reason: Skip dynamic_stacking when use_bag_holdout is enabled. (use_bag_holdout=True)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 3600s
AutoGluon will save models to "c:\Users\Marte\Documents\SCHOOL\2024_2026_KUL\2025-2026\SEMESTER 2\Advanced Analytics in a Big Data World

                    model  score_test  score_val          eval_metric  \
0       LightGBMXT_BAG_L1   -0.630518  -0.630518  mean_absolute_error   
1     WeightedEnsemble_L3   -0.630518  -0.630518  mean_absolute_error   
2     WeightedEnsemble_L2   -0.630518  -0.630518  mean_absolute_error   
3         LightGBM_BAG_L1   -0.631279  -0.631279  mean_absolute_error   
4  RandomForestMSE_BAG_L1   -0.640931  -0.640931  mean_absolute_error   

   pred_time_test  pred_time_val     fit_time  pred_time_test_marginal  \
0        0.395365       1.067187    22.038187                 0.395365   
1        0.434760       1.069578    22.150553                 0.039395   
2        0.435544       1.069602    22.158854                 0.040179   
3        0.262971       0.935515    21.241397                 0.262971   
4        1.818734       1.854088  4218.192598                 1.818734   

   pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  \
0                1.067187          22.038187

In [28]:
# Two-stage validatie: soft combinatie en hard threshold search

proba_val_buy = predictor_cls.predict_proba(X_val_cls)[1]
pred_val_amount_buyers = np.expm1(predictor_buyers.predict(X_val_cls))
pred_val_amount_buyers = np.clip(np.asarray(pred_val_amount_buyers), 0, None)

pred_val_soft = np.clip(np.asarray(proba_val_buy) * pred_val_amount_buyers, 0, None)
mae_two_stage_soft = mean_absolute_error(y_val_true, pred_val_soft)
sp_two_stage_soft = safe_spearman(y_val_true, pred_val_soft)

threshold_rows = []
for threshold in np.arange(0.1, 0.91, 0.05):
    pred_val_hard = np.where(np.asarray(proba_val_buy) >= threshold, pred_val_amount_buyers, 0.0)
    threshold_rows.append({
        "threshold": round(float(threshold), 2),
        "MAE": mean_absolute_error(y_val_true, pred_val_hard),
        "Spearman": safe_spearman(y_val_true, pred_val_hard)
    })

threshold_results = pd.DataFrame(threshold_rows)
best_threshold_row = threshold_results.sort_values(["MAE", "Spearman"], ascending=[True, False]).iloc[0]
best_threshold = float(best_threshold_row["threshold"])
pred_val_hard_best = np.where(np.asarray(proba_val_buy) >= best_threshold, pred_val_amount_buyers, 0.0)

mae_two_stage_hard = mean_absolute_error(y_val_true, pred_val_hard_best)
sp_two_stage_hard = safe_spearman(y_val_true, pred_val_hard_best)

two_stage_results = pd.DataFrame([
    {"model_name": "TwoStage_soft", "MAE": mae_two_stage_soft, "Spearman": sp_two_stage_soft},
    {"model_name": "TwoStage_hard_threshold", "MAE": mae_two_stage_hard, "Spearman": sp_two_stage_hard, "threshold": best_threshold}
])

print("Soft two-stage")
print(two_stage_results.iloc[0])
print()
print("Best hard-threshold two-stage")
print(two_stage_results.iloc[1])

threshold_results.sort_values("MAE").head(10)

Soft two-stage
model_name    TwoStage_soft
MAE               71.348229
Spearman           0.408387
threshold               NaN
Name: 0, dtype: object

Best hard-threshold two-stage
model_name    TwoStage_hard_threshold
MAE                          63.02577
Spearman                     0.365703
threshold                        0.65
Name: 1, dtype: object


,threshold,MAE,Spearman
11,0.65,63.025770,0.365703
10,0.60,63.162600,0.384678
12,0.70,63.534621,0.339126
9,0.55,63.583954,0.400670
13,0.75,64.060268,0.315262
8,0.50,64.525408,0.415539
14,0.80,64.871646,0.283242
15,0.85,65.901043,0.240156
7,0.45,65.931340,0.420031
16,0.90,67.366023,0.185382


In [29]:
# Voeg two-stage toe aan het modeloverzicht

comparison_with_two_stage = pd.concat([
    comparison_with_ensemble,
    two_stage_results[["model_name", "MAE", "Spearman"]]
], ignore_index=True)

comparison_with_two_stage.to_csv("autogluon_full_comparison_with_twostage.csv", index=False)
comparison_with_two_stage.sort_values(["MAE", "Spearman"], ascending=[True, False])

,model_name,MAE,Spearman
2,AutoGluon_base_log_ensemble,61.788168,0.378258
1,AutoGluon_log1p,61.856358,0.395572
0,AutoGluon_base,61.870484,0.377204
4,TwoStage_hard_threshold,63.025770,0.365703
3,TwoStage_soft,71.348229,0.408387


In [ ]:
# Submission voor de soft two-stage voorspelling

proba_test_buy = predictor_cls.predict_proba(X_test_ag)[1]
pred_test_amount_buyers = np.expm1(predictor_buyers.predict(X_test_ag))
pred_test_amount_buyers = np.clip(np.asarray(pred_test_amount_buyers), 0, None)

pred_test_soft = np.clip(np.asarray(proba_test_buy) * pred_test_amount_buyers, 0, None)

submission_two_stage_soft = pd.DataFrame({
    "cust_id": test_cust_ids,
    "predicted_revenue_2018_2019": pred_test_soft
})

submission_two_stage_soft_name = "submission_autogluon_two_stage_soft.csv"
submission_two_stage_soft.to_csv(submission_two_stage_soft_name, index=False)

print("Saved:", submission_two_stage_soft_name)
print("Prediction summary")
print("min :", pred_test_soft.min())
print("max :", pred_test_soft.max())
print("mean:", pred_test_soft.mean())

submission_two_stage_soft.head()

In [ ]:
# Submission voor de beste hard-threshold two-stage voorspelling

pred_test_hard = np.where(np.asarray(proba_test_buy) >= best_threshold, pred_test_amount_buyers, 0.0)
pred_test_hard = np.clip(np.asarray(pred_test_hard), 0, None)

submission_two_stage_hard = pd.DataFrame({
    "cust_id": test_cust_ids,
    "predicted_revenue_2018_2019": pred_test_hard
})

submission_two_stage_hard_name = "submission_autogluon_two_stage_hard.csv"
submission_two_stage_hard.to_csv(submission_two_stage_hard_name, index=False)

print("Saved:", submission_two_stage_hard_name)
print("Threshold:", best_threshold)
print("Prediction summary")
print("min :", pred_test_hard.min())
print("max :", pred_test_hard.max())
print("mean:", pred_test_hard.mean())

submission_two_stage_hard.head()